# VALORA · Anexo técnico reproducible
Máster en Data Science, Big Data y Business Analytics. Fuente: [idealista18](https://github.com/paezha/idealista18), Rey-Blanco et al. (2024), [DOI](https://doi.org/10.1177/23998083241242844), [ODbL](https://github.com/paezha/idealista18/blob/master/LICENSE.md).

**Objetivo:** predecir precios de oferta de 2018. La proyección mediante índice posterior es un escenario sin validación externa. El notebook ejecuta el motor del repositorio, por lo que solo existe una metodología final.


## 1. Entorno en Colab
Sube a la vez el ZIP del proyecto mejorado y data(2).zip. Los seis archivos Sale y Polygons terminan dentro de data/. Ejecuta las celdas en orden.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, subprocess, sys
BASE=Path('/content/tfm_valora');BASE.mkdir(parents=True,exist_ok=True)
files.upload()
project=next((z for z in Path('/content').glob('*.zip') if any(n.endswith('/entrenar.py') or n=='entrenar.py' for n in zipfile.ZipFile(z).namelist())),None)
datazip=next((z for z in Path('/content').glob('*.zip') if z!=project and 'data' in z.name.lower()),None)
assert project and datazip, 'Debes subir los dos ZIP'
with zipfile.ZipFile(project) as z:z.extractall(BASE)
ROOT=next((folder for folder in BASE.iterdir() if (folder/'entrenar.py').exists()),None)
assert ROOT, 'No se encuentra entrenar.py dentro del ZIP del proyecto'
with zipfile.ZipFile(datazip) as z:
    for member in z.namelist():
        if member.startswith('data/') and member.endswith('.rda'):
            dest=ROOT/member;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(z.read(member))
assert (ROOT/'data/Madrid_Sale.rda').exists()
subprocess.run([sys.executable,'-m','pip','-q','install','-r',str(ROOT/'requirements.txt')],check=True)
print('Listo:',ROOT)


## 2. Calidad, partición y entrenamiento
Se eliminan duplicados por ASSETID y trimestre, y filas fuera de los rangos de precio, superficie y ubicación declarados en modelado.py. ASSETID nunca cruza las particiones de train, calibración y test. PRICE y UNITPRICE son objetivos, no predictores. Los modelos de cuantiles 10, 50 y 90 predicen log(€/m²) y se convierten a euros; el intervalo se calibra en un conjunto separado.


In [ ]:
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt
subprocess.run([sys.executable,'entrenar.py'],cwd=ROOT,check=True)
summary=json.loads((ROOT/'artefactos/resumen.json').read_text())
flow={city:json.loads((ROOT/'artefactos'/f'{city.lower()}_meta.json').read_text())['flujo'] for city in summary}
display(pd.DataFrame(flow).T)


## 3. Baselines y test final
Las medianas de €/m² por ciudad y distrito se ajustan solo con train. El test mide los tres métodos sobre exactamente las mismas viviendas, sin seleccionar ganadores mirando sus métricas. El modelo servido es el modelo entrenado solo con train: nunca se reentrena con test después de publicar sus resultados.


In [ ]:
rows=[{'ciudad':city,'metodo':name,**values[name]} for city,values in summary.items() for name in ('baseline_ciudad','baseline_distrito','modelo')]
results=pd.DataFrame(rows)
display(results.round(2))
display(pd.DataFrame({city:{'n_test':r['n_test'],'cobertura_2018':r['cobertura_2018'],'ancho_mediano_eur':r['ancho_mediano_2018_eur']} for city,r in summary.items()}).T)
results.pivot(index='ciudad',columns='metodo',values='mae_eur').plot.bar(figsize=(9,4),rot=0,ylabel='MAE en € · test 2018')
plt.tight_layout();plt.show()


## 4. Robustez espacial y temporal
La prueba espacial entrena OTRO modelo que no ha visto determinados distritos enteros. La temporal entrena OTRO modelo con Q1–Q3 y evalúa viviendas nuevas de Q4. Son pruebas distintas del test principal y ninguna justifica precisión en 2026. El error por distrito procede exclusivamente del test principal.


In [ ]:
display(pd.DataFrame([{'ciudad':c,**r['espacial_distritos_nuevos']} for c,r in summary.items()]))
display(pd.DataFrame([{'ciudad':c,**r.get('temporal_q4_nuevos',{})} for c,r in summary.items()]))
for city in summary:
    error=pd.DataFrame(json.loads((ROOT/'artefactos'/f'{city.lower()}_error_distrito.json').read_text())).T.sort_values('mae_eur')
    print(city);display(pd.concat([error.head(3),error.tail(3)]))


## 4.1. Ablación y uso real de la interfaz
El modelo sin variables espaciales muestra la aportación de la localización. La simulación de los campos de la app sustituye atributos no solicitados por medianas de TRAIN: su MAE es más representativo del uso real que el test con todos los atributos observados. La cobertura calibrada principal no se traslada automáticamente a esa simulación.


In [ ]:
display(pd.DataFrame([
    {'ciudad':city,'modelo_completo_mae':v['modelo']['mae_eur'],
     'sin_localizacion_mae':v['ablacion_sin_localizacion']['mae_eur'],
     'solo_campos_app_mae':v['simulacion_campos_app']['mae_eur']}
    for city,v in summary.items()]))


## 5. Interpretabilidad
Las contribuciones nativas de los árboles son aportaciones SHAP al logaritmo del precio unitario. Sus magnitudes muestran qué variables usa el modelo, no el efecto causal de cambiar una característica de una vivienda.


In [ ]:
import xgboost as xgb
sys.path.insert(0,str(ROOT))
from geografia import leer_rda
from modelado import limpiar, preparar
meta=json.loads((ROOT/'artefactos/madrid_meta.json').read_text())
raw=pd.DataFrame(leer_rda('Madrid_Sale.rda','Madrid_Sale')).drop(columns='geometry',errors='ignore')
raw,_=limpiar(raw)
X=preparar(raw.sample(500,random_state=42),meta['features'])
booster=xgb.Booster(model_file=str(ROOT/'artefactos/madrid_q50.ubj'))
contrib=booster.predict(xgb.DMatrix(X),pred_contribs=True)
assert contrib.shape[1]==len(X.columns)+1
imp=pd.Series(np.abs(contrib[:,:-1]).mean(axis=0),index=X.columns).sort_values(ascending=False)
display(imp.head(15).rename('SHAP absoluto medio, escala log €/m²'))
imp.head(12).sort_values().plot.barh(figsize=(8,5));plt.tight_layout();plt.show()


## 6. Discusión y entrega
Copia solo las cifras de resumen.json a la memoria. Explica la comparación con las referencias, la cobertura observada y las limitaciones: anuncios en lugar de ventas, coordenadas aproximadas, atributos faltantes y ausencia de validación de la proyección actual. Exporta este notebook ejecutado a HTML como anexo; la memoria de negocio tiene un límite de 20 caras según la guía UCM.
